In [ ]:
%%bash

#pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


1/The objective is to predict the sale price of residential homes in Ames, Iowa. Given various features describing the properties, we aim to accurately estimate the final selling price in dollars (SalePrice variable)

2/This is a supervised learning regression problem. We have labeled training data where each house has known features and a corresponding sale price. The model will learn the relationship between the input features and the continuous output variable (SalePrice).

3/Performance should be measured using Root Mean Squared Logarithmic Error (RMSLE), which is the official Kaggle competition metric because it handles the wide range of house prices well and penalizes relative errors rather than absolute dollar amounts.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load the data
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# 2. Check size and type of data
print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
print("\n" + "="*50)
print("Training data info:")
print(train_data.info())
print("\n" + "="*50)
print("First few rows:")
print(train_data.head())

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
# Analyze feature types
print("Data types distribution:")
print(train_data.dtypes.value_counts())
print("\n" + "="*50)

# Separate features by type
numeric_features = train_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = train_data.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

# Check some examples
print("\n" + "="*50)
print("Sample categorical values:")
for col in categorical_features[:5]:
    print(f"\n{col}: {train_data[col].unique()[:10]}")

In [ ]:
# 2. Features with missing values
print("Features with missing values:")
missing = train_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
print(f"\nTotal features with missing values: {len(missing)}")
print(f"Percentage of missing values:\n{(missing / len(train_data) * 100).round(2)}")

In [ ]:
def drop_high_missing_features(train_df, test_df, threshold=0.8):
    """
    Drop features with missing values above threshold
    
    Parameters:
    -----------
    train_df : DataFrame
    test_df : DataFrame
    threshold : float (default 0.8 = 80%)
    
    Returns:
    --------
    train_df, test_df, dropped_features
    """
    # Calculate missing percentage on training data
    missing_pct = train_df.isnull().sum() / len(train_df)
    
    # Identify features to drop
    drop_features = missing_pct[missing_pct > threshold].index.tolist()
    
    # Drop from both datasets
    train_df = train_df.drop(columns=drop_features)
    test_df = test_df.drop(columns=drop_features)
    
    print(f"Dropped {len(drop_features)} features with >{threshold*100}% missing:")
    print(drop_features)
    
    return train_df, test_df, drop_features

# Usage:
train_data, test_data, dropped = drop_high_missing_features(train_data, test_data, threshold=0.8)

In [ ]:
def detect_outliers_iqr(df, features, exclude_zero_inflated=True):
    """
    Detect outliers using IQR method
    Excludes zero-inflated features by default
    """
    outlier_summary = []
    
    for feature in features:
        data = df[feature].dropna()
        
        # Skip zero-inflated features (>50% zeros)
        if exclude_zero_inflated:
            zero_pct = (data == 0).sum() / len(data)
            if zero_pct > 0.5:
                continue  # Skip this feature
        
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        
        # Skip if IQR is 0 (all values same or mostly same)
        if IQR == 0:
            continue
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = data[(data < lower_bound) | (data > upper_bound)]
        outlier_count = len(outliers)
        outlier_pct = (outlier_count / len(data)) * 100
        
        outlier_summary.append({
            'Feature': feature,
            'Outlier_Count': outlier_count,
            'Outlier_Pct': outlier_pct,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })
    
    return pd.DataFrame(outlier_summary).sort_values('Outlier_Pct', ascending=False)


# Usage - only check continuous features
continuous_features = [
    'LotFrontage', 'LotArea', 'MasVnrArea',
    'BsmtFinSF1', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 
    'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF'
]

outlier_df = detect_outliers_iqr(train_data, continuous_features, exclude_zero_inflated=True)
print("Features with most outliers (excluding zero-inflated):")
print(outlier_df.head(10))

In [ ]:
# 3. Outliers detection
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot for numeric features
numeric_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
fig, axes = plt.subplots(6, 6, figsize=(20, 15))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols[:36]):
    axes[idx].boxplot(train_data[col].dropna())
    axes[idx].set_title(col, fontsize=8)
    axes[idx].tick_params(labelsize=6)

plt.tight_layout()
plt.show()



# Scatter plot to identify outliers
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(train_data['GrLivArea'], train_data['SalePrice'], alpha=0.5)
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')
plt.title('SalePrice vs GrLivArea')

plt.subplot(1, 2, 2)
plt.scatter(train_data['LotArea'], train_data['SalePrice'], alpha=0.5)
plt.xlabel('LotArea')
plt.ylabel('SalePrice')
plt.title('SalePrice vs LotArea')
plt.tight_layout()
plt.show()

In [ ]:
def remove_problematic_outliers(train_df):
    """
    Remove observations with inconsistent GrLivArea and SalePrice patterns.
    
    Identifies houses with exceptionally large living areas but disproportionately
    low sale prices, which likely represent data quality issues or special 
    circumstances that could negatively impact model training.
    
    Parameters:
    -----------
    train_df : DataFrame
        Training dataset
    
    Returns:
    --------
    DataFrame with problematic outliers removed
    """
    initial = len(train_df)
    
    # Identify and remove inconsistent observations
    train_df = train_df[~((train_df['GrLivArea'] > 4000) & 
                          (train_df['SalePrice'] < 300000))]
    
    removed = initial - len(train_df)
    print(f"Removed {removed} observations with inconsistent patterns")
    print(f"Remaining training samples: {len(train_df)}")
    
    return train_df

# Apply to training data only
train_data = remove_problematic_outliers(train_data)

In [ ]:
# 4. Feature usefulness analysis
from sklearn.ensemble import RandomForestRegressor

# Prepare data (simple approach - drop nulls and encode categoricals)
train_numeric = train_data.select_dtypes(include=['int64', 'float64']).dropna(axis=1)
X = train_numeric.drop(['Id', 'SalePrice'], axis=1, errors='ignore')
y = train_data['SalePrice']

# Remove rows with missing target
mask = y.notna()
X = X[mask]
y = y[mask]

# Feature importance using Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 most useful features:")
print(feature_importance.head(15))
print("\nBottom 10 potentially useless features:")
print(feature_importance.tail(10))

# Visualize
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Target distribution analysis
print("SalePrice statistics:")
print(train_data['SalePrice'].describe())
print(f"\nSkewness: {train_data['SalePrice'].skew():.2f}")
print(f"Kurtosis: {train_data['SalePrice'].kurtosis():.2f}")

# Visualize target distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(train_data['SalePrice'], bins=50, edgecolor='black')
axes[0].set_xlabel('SalePrice')
axes[0].set_ylabel('Frequency')
axes[0].set_title('SalePrice Distribution')

# Log-transformed distribution
axes[1].hist(np.log1p(train_data['SalePrice']), bins=50, edgecolor='black')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-transformed SalePrice Distribution')

# Q-Q plot
from scipy import stats
stats.probplot(train_data['SalePrice'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

In [ ]:
# 6. Data visualization 
# Relationship between key features and target
key_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(key_features):
    axes[idx].scatter(train_data[feature], train_data['SalePrice'], alpha=0.5)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('SalePrice')
    axes[idx].set_title(f'SalePrice vs {feature}')

plt.tight_layout()
plt.show()

# Categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cat_features = ['Neighborhood', 'HouseStyle', 'ExterQual', 'KitchenQual']

for idx, feature in enumerate(cat_features):
    ax = axes[idx // 2, idx % 2]
    train_data.boxplot(column='SalePrice', by=feature, ax=ax)
    ax.set_xlabel(feature)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'SalePrice by {feature}')
    plt.sca(ax)
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 7. Correlation analysis
# Correlation matrix for numeric features
numeric_data = train_data.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numeric_data.corr()

# Full correlation heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - All Numeric Features')
plt.tight_layout()
plt.show()

# Top correlations with SalePrice
print("Top 15 features correlated with SalePrice:")
price_corr = correlation_matrix['SalePrice'].sort_values(ascending=False)
print(price_corr.head(16))  # 16 because first is SalePrice itself

# Visualize top correlations
plt.figure(figsize=(8, 10))
price_corr[1:16].plot(kind='barh')
plt.title('Top 15 Features Correlated with SalePrice')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

# Check multicollinearity among top features
top_features = price_corr[1:11].index.tolist() + ['SalePrice']
plt.figure(figsize=(10, 8))
sns.heatmap(train_data[top_features].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix - Top 10 Features + SalePrice')
plt.tight_layout()
plt.show()

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd
class MissingValueHandler(BaseEstimator, TransformerMixin):
    """
    Handles missing values based on feature type and domain knowledge.
    """
    def __init__(self):
        self.fill_values = {}
        
        # Categorical features where missing means "feature doesn't exist"
        self.none_features = [
            'Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
            'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType',
            'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence',
            'MiscFeature'
        ]
        
        # Numerical area features where missing = 0 (feature doesn't exist)
        self.zero_features = [
            'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
            'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars',
            'GarageArea'
        ]
    
    def fit(self, X, y=None):
        """
        Learn the fill values for each column based on training data.
        """
        for col in X.columns:
            # Categorical features where missing = "doesn't exist"
            if col in self.none_features:
                self.fill_values[col] = 'None'
            
            # Numerical area features where missing = 0
            elif col in self.zero_features:
                self.fill_values[col] = 0
            
            # Other categorical features: use mode
            elif X[col].dtype == 'object':
                mode_values = X[col].mode()
                self.fill_values[col] = mode_values[0] if len(mode_values) > 0 else 'Missing'
            
            # Other numerical features: use median
            else:
                self.fill_values[col] = X[col].median()
        
        return self
    
    def transform(self, X):
        """
        Fill missing values using the learned fill values.
        Also handles any remaining missing values in test data.
        """
        X = X.copy()
        
        # Apply learned fill values
        for col, fill_value in self.fill_values.items():
            if col in X.columns:
                X[col] = X[col].fillna(fill_value)
        
        # Handle any REMAINING missing values (edge cases in test data)
        for col in X.columns:
            if X[col].isnull().sum() > 0:
                if X[col].dtype == 'object':
                    # Categorical: fill with 'Missing'
                    X[col] = X[col].fillna('Missing')
                else:
                    # Numerical: fill with 0 or median
                    if col in self.zero_features:
                        X[col] = X[col].fillna(0)
                    else:
                        X[col] = X[col].fillna(X[col].median())
        
        return X


# Re-run the cleaning process:
print("Step 1: Data Cleaning")
print("="*50)

# Drop Id column
train_clean = train_data.drop('Id', axis=1)
test_ids = test_data['Id']
test_clean = test_data.drop('Id', axis=1)

print(f"Original training shape: {train_data.shape}")
print(f"After dropping Id: {train_clean.shape}")

# Separate target
y_train = train_clean['SalePrice'].copy()
X_train = train_clean.drop('SalePrice', axis=1)

print(f"\nFeatures shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")

# Apply log transformation to target
y_train_log = np.log1p(y_train)
print(f"\nTarget transformed to log scale")
print(f"Original skewness: {y_train.skew():.2f}")
print(f"Log-transformed skewness: {y_train_log.skew():.2f}")

# Initialize and apply missing value handler
print("\n" + "="*50)
print("Handling Missing Values")
print("="*50)

missing_handler = MissingValueHandler()

# Fit on training data and transform both
X_train_clean = missing_handler.fit_transform(X_train)
X_test_clean = missing_handler.transform(test_clean)

print(f"\nMissing values after cleaning:")
print(f"Training: {X_train_clean.isnull().sum().sum()}")
print(f"Test: {X_test_clean.isnull().sum().sum()}")

print(f"\nFinal shapes:")
print(f"Training: {X_train_clean.shape}")
print(f"Test: {X_test_clean.shape}")

print("\n✓ Data cleaning complete!")

# Optional: Check which columns had remaining missing values
if X_test_clean.isnull().sum().sum() > 0:
    print("\nColumns with missing values in test (before final fix):")
    print(X_test_clean.isnull().sum()[X_test_clean.isnull().sum() > 0])

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew  # ← ADD THIS
import numpy as np
import pandas as pd
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Applies feature engineering transformations
    """
    def __init__(self, skew_threshold=0.75, zero_threshold=0.3):
        self.skew_threshold = skew_threshold
        self.zero_threshold = zero_threshold
        
        self.drop_features = ['PoolArea', '3SsnPorch', 'LowQualFinSF', 'MiscVal']
        self.binary_features = ['BsmtFinSF2', 'EnclosedPorch', 'WoodDeckSF']
        
        # Features to ALWAYS exclude from log transformation
        self.exclude_from_log = [
            'MSSubClass',      # Categorical codes, not continuous
            'KitchenAbvGr',    # Count feature, mostly 1
            'BedroomAbvGr',    # Count feature
            'FullBath',        # Count feature
            'HalfBath',        # Count feature
            'BsmtFullBath',    # Count feature
            'BsmtHalfBath',    # Count feature
            'Fireplaces',      # Count feature
            'GarageCars',      # Count feature
            'OverallQual',     # Ordinal rating (1-10)
            'OverallCond',     # Ordinal rating (1-10)
            'YearBuilt',       # Year (not skewed)
            'YearRemodAdd',    # Year (not skewed)
            'GarageYrBlt',     # Year (not skewed)
            'MoSold',          # Month (1-12)
            'YrSold'           # Year (not skewed)
        ]
        
        self.log_features = []
        
    def fit(self, X, y=None):
        X_temp = X.copy()
        X_temp = X_temp.drop(columns=self.drop_features, errors='ignore')
        
        for feature in self.binary_features:
            if feature in X_temp.columns:
                X_temp = X_temp.drop(columns=[feature])
        
        numeric_features = X_temp.select_dtypes(include=['int64', 'float64']).columns
        
        self.log_features = []
        for feature in numeric_features:
            # Skip excluded features
            if feature in self.exclude_from_log:
                continue
                
            data = X_temp[feature].dropna()
            feature_skew = abs(skew(data))
            zero_pct = (data == 0).sum() / len(data)
            
            # Only log-transform if skewed AND not zero-inflated
            if feature_skew > self.skew_threshold and zero_pct < self.zero_threshold:
                self.log_features.append(feature)
        
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=self.drop_features, errors='ignore')
        
        for feature in self.binary_features:
            if feature in X.columns:
                X[f'Has{feature}'] = (X[feature] > 0).astype(int)
                X = X.drop(columns=[feature])
        
        for feature in self.log_features:
            if feature in X.columns:
                X[feature] = np.log1p(X[feature])
        
        return X


# Re-run:
print("Feature Engineering (Corrected)")
print("="*70)

feature_engineer = FeatureEngineer(skew_threshold=0.75, zero_threshold=0.3)
X_train_engineered = feature_engineer.fit_transform(X_train_clean)
X_test_engineered = feature_engineer.transform(X_test_clean)

print(f"Original shape: {X_train_clean.shape}")
print(f"After engineering: {X_train_engineered.shape}")

print(f"\nTransformations applied:")
print(f"  - Dropped: {feature_engineer.drop_features}")
print(f"  - Binary indicators created: {[f'Has{f}' for f in feature_engineer.binary_features]}")
print(f"  - Log transformed ({len(feature_engineer.log_features)} features): {feature_engineer.log_features}")

print("\n✓ Feature engineering complete!")

In [ ]:
def visualize_log_transformation(X_before, X_after, features):
    """
    Visualize distribution before/after log transformation
    """
    from scipy.stats import skew
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(len(features), 2, figsize=(14, 12))
    
    for idx, feature in enumerate(features):
        before_data = X_before[feature]
        after_data = X_after[feature]
        before_skew = skew(before_data.dropna())
        after_skew = skew(after_data.dropna())
        
        # Before
        axes[idx, 0].hist(before_data.dropna(), bins=50, edgecolor='black', alpha=0.7, color='skyblue')
        axes[idx, 0].set_title(f'{feature} - BEFORE (skew={before_skew:.2f})')
        axes[idx, 0].set_ylabel('Frequency')
        
        # After
        axes[idx, 1].hist(after_data.dropna(), bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
        axes[idx, 1].set_title(f'{feature} - AFTER LOG (skew={after_skew:.2f})')
        axes[idx, 1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

# Use:
visualize_log_transformation(X_train_clean, X_train_engineered, 
                            ['LotFrontage', 'LotArea', 'BsmtUnfSF', '1stFlrSF', 'GrLivArea'])

In [ ]:
from scipy.stats import skew

def analyze_feature_types(X_train):
    """
    Analyzes and categorizes features into:
    - Numerical continuous
    - Numerical discrete (potential ordinal)
    - Categorical
    
    Returns summaries of zero-inflation, skewness, and unique values.
    """
    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
    
    print("Feature Type Analysis")
    print("="*70)
    print(f"Numerical features: {len(numeric_features)}")
    print(f"Categorical features: {len(categorical_features)}")
    
    # Analyze numerical features
    analysis = []
    for feature in numeric_features:
        data = X_train[feature]
        analysis.append({
            'Feature': feature,
            'Unique': data.nunique(),
            'Zeros_%': (data == 0).sum() / len(data) * 100,
            'Skewness': skew(data.dropna())
        })
    
    analysis_df = pd.DataFrame(analysis)
    
    # Show features with few unique values (potentially ordinal)
    print("\n1. Numerical features with ≤15 unique values (potentially ordinal):")
    few_unique = analysis_df[analysis_df['Unique'] <= 15].sort_values('Unique')
    if len(few_unique) > 0:
        print(few_unique.to_string(index=False))
    
    # Show zero-inflated features
    print("\n2. Zero-inflated features (>50% zeros):")
    zero_inflated = analysis_df[analysis_df['Zeros_%'] > 50].sort_values('Zeros_%', ascending=False)
    if len(zero_inflated) > 0:
        print(zero_inflated[['Feature', 'Zeros_%']].to_string(index=False))
    
    # Show highly skewed features
    print("\n3. Highly skewed features (|skewness| > 2):")
    highly_skewed = analysis_df[analysis_df['Skewness'].abs() > 2].sort_values('Skewness', key=abs, ascending=False)
    if len(highly_skewed) > 0:
        print(highly_skewed[['Feature', 'Skewness']].to_string(index=False))
    
    return analysis_df, categorical_features

# Run analysis
analysis_df, categorical_features = analyze_feature_types(X_train_engineered)

In [ ]:
def check_multicollinearity(X_train, threshold=0.9):
    """
    Identifies highly correlated feature pairs (potential redundancy).
    """
    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    corr_matrix = X_train[numeric_features].corr()
    
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                high_corr_pairs.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    print(f"\nHighly Correlated Pairs (|correlation| > {threshold}):")
    print("="*70)
    if high_corr_pairs:
        high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
        print(high_corr_df.to_string(index=False))
    else:
        print("No highly correlated pairs found.")
    
    return high_corr_pairs

# Check correlations
high_corr = check_multicollinearity(X_train_engineered, threshold=0.9)

In [ ]:
def identify_categorical_encoding_strategy(X_train, categorical_features):
    """
    Identifies which categorical features need:
    - Ordinal encoding (ordered categories)
    - One-hot encoding (unordered categories)
    """
    print("\nCategorical Feature Encoding Strategy")
    print("="*70)
    
    # Features that should be treated as categorical (even if numeric)
    numeric_categorical = ['MSSubClass']  # Building type codes
    
    # Features with natural order (will use ordinal encoding later)
    ordinal_features = [
        'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
        'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
        'FireplaceQu', 'GarageQual', 'GarageCond', 'GarageFinish',
        'PoolQC', 'Fence', 'LotShape', 'LandSlope', 'Functional',
        'PavedDrive', 'Utilities'
    ]
    
    # All other categorical features (will use one-hot encoding)
    nominal_features = [f for f in categorical_features if f not in ordinal_features]
    
    print(f"Ordinal features (natural order): {len(ordinal_features)}")
    print(f"Nominal features (no order): {len(nominal_features)}")
    print(f"Numeric features to treat as categorical: {numeric_categorical}")
    
    return ordinal_features, nominal_features, numeric_categorical

ordinal_features, nominal_features, numeric_categorical = identify_categorical_encoding_strategy(
    X_train_engineered, 
    categorical_features
)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class OrdinalEncoder(BaseEstimator, TransformerMixin):
    """
    Encodes ordinal categorical features with meaningful order.
    """
    def __init__(self):
        # Define ordinal mappings
        self.ordinal_mappings = {
            # Quality ratings (Ex > Gd > TA > Fa > Po > None)
            'ExterQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'ExterCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'BsmtQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'BsmtCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'FireplaceQu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'GarageQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'GarageCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            'PoolQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
            
            # Basement exposure
            'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
            
            # Basement finish type
            'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
            'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
            
            # Garage finish
            'GarageFinish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0},
            
            # Fence quality
            'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0},
            
            # Lot shape
            'LotShape': {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1},
            
            # Land slope
            'LandSlope': {'Gtl': 3, 'Mod': 2, 'Sev': 1},
            
            # Functionality
            'Functional': {'Typ': 8, 'Min1': 7, 'Min2': 6, 'Mod': 5, 'Maj1': 4, 'Maj2': 3, 'Sev': 2, 'Sal': 1},
            
            # Paved drive
            'PavedDrive': {'Y': 3, 'P': 2, 'N': 1},
            
            # Utilities
            'Utilities': {'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1}
        }
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col, mapping in self.ordinal_mappings.items():
            if col in X.columns:
                # Map values, keep unmapped as NaN then fill with 0
                X[col] = X[col].map(mapping).fillna(0).astype(int)
        
        return X


print("Ordinal Encoding")
print("="*70)

# Apply ordinal encoding
ordinal_encoder = OrdinalEncoder()
X_train_ordinal = ordinal_encoder.fit_transform(X_train_engineered)
X_test_ordinal = ordinal_encoder.transform(X_test_engineered)

print(f"Shape after ordinal encoding: {X_train_ordinal.shape}")
print(f"Ordinal features encoded: {len(ordinal_encoder.ordinal_mappings)}")
print("\n✓ Ordinal encoding complete!")

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd

def apply_onehot_encoding(X_train, X_test):
    """
    Apply one-hot encoding to nominal categorical features.
    """
    # Identify categorical columns (object dtype)
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    
    # Also treat MSSubClass as categorical
    if 'MSSubClass' in X_train.columns:
        X_train['MSSubClass'] = X_train['MSSubClass'].astype(str)
        X_test['MSSubClass'] = X_test['MSSubClass'].astype(str)
        if 'MSSubClass' not in categorical_cols:
            categorical_cols.append('MSSubClass')
    
    # Get numerical columns
    numerical_cols = [col for col in X_train.columns if col not in categorical_cols]
    
    print(f"Numerical features: {len(numerical_cols)}")
    print(f"Categorical features to one-hot encode: {len(categorical_cols)}")
    
    # Create column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numerical_cols),
            ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), 
             categorical_cols)
        ],
        verbose_feature_names_out=False
    )
    
    # Fit and transform
    X_train_encoded = preprocessor.fit_transform(X_train)
    X_test_encoded = preprocessor.transform(X_test)
    
    # Get feature names
    feature_names = numerical_cols.copy()
    if categorical_cols:
        cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
        feature_names.extend(cat_features)
    
    # Convert to DataFrame
    X_train_final = pd.DataFrame(X_train_encoded, columns=feature_names, index=X_train.index)
    X_test_final = pd.DataFrame(X_test_encoded, columns=feature_names, index=X_test.index)
    
    return X_train_final, X_test_final, preprocessor


print("\nOne-Hot Encoding")
print("="*70)

# Apply one-hot encoding
X_train_full, X_test_full, onehot_preprocessor = apply_onehot_encoding(X_train_ordinal, X_test_ordinal)

print(f"\nFinal shape after all encoding: {X_train_full.shape}")
print(f"Total features: {X_train_full.shape[1]}")
print("\n✓ One-hot encoding complete!")

# Summary of all preprocessing steps
print("\n" + "="*70)
print("COMPLETE PREPROCESSING SUMMARY:")
print("="*70)
print(f"Original features: {X_train.shape[1]}")
print(f"After feature engineering: {X_train_engineered.shape[1]}")
print(f"After ordinal encoding: {X_train_ordinal.shape[1]}")
print(f"After one-hot encoding: {X_train_full.shape[1]}")
print("\nAll preprocessing transformations complete! ✅")

In [ ]:
def create_all_feature_combinations():
    """
    Create all feature combination datasets for model training.
    """
    combinations = {}
    
    # 1. Numeric only (no engineering)
    X_train_numeric_raw = X_train_clean.select_dtypes(include=['int64', 'float64'])
    X_test_numeric_raw = X_test_clean.select_dtypes(include=['int64', 'float64'])
    combinations['1_Numeric_NoEngineering'] = {
        'X_train': X_train_numeric_raw,
        'X_test': X_test_numeric_raw,
        'description': 'Only numerical features, no transformations'
    }
    
    # 2. Numeric only (with engineering)
    X_train_numeric_eng = X_train_engineered.select_dtypes(include=['int64', 'float64'])
    X_test_numeric_eng = X_test_engineered.select_dtypes(include=['int64', 'float64'])
    combinations['2_Numeric_WithEngineering'] = {
        'X_train': X_train_numeric_eng,
        'X_test': X_test_numeric_eng,
        'description': 'Numerical features with log transforms and binary indicators'
    }
    
    # 3. Numeric + Ordinal (with engineering)
    X_train_with_ordinal = X_train_ordinal.select_dtypes(include=['int64', 'float64'])
    X_test_with_ordinal = X_test_ordinal.select_dtypes(include=['int64', 'float64'])
    combinations['3_Numeric+Ordinal'] = {
        'X_train': X_train_with_ordinal,
        'X_test': X_test_with_ordinal,
        'description': 'Numerical + ordinal categorical features'
    }
    
    # 4. All features (numeric + ordinal + one-hot nominal)
    combinations['4_AllFeatures'] = {
        'X_train': X_train_full,
        'X_test': X_test_full,
        'description': 'Complete dataset with all features encoded'
    }
    
    # Print summary
    print("Feature Combinations Created:")
    print("="*70)
    for name, data in combinations.items():
        print(f"\n{name}:")
        print(f"  Features: {data['X_train'].shape[1]}")
        print(f"  Description: {data['description']}")
    
    return combinations


# Create all combinations
feature_combinations = create_all_feature_combinations()

print("\n" + "="*70)
print("✓ All feature combinations ready for model training!")

### Feature engineering

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class AdvancedFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Creates high-value engineered features (non-redundant, strong predictors).
    Apply AFTER basic FeatureEngineer and ordinal encoding, BEFORE one-hot encoding.
    """
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # === TOTALS & AGGREGATES ===
        
        # 1. Total Square Footage (CRITICAL)
        if all(col in X.columns for col in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']):
            X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
        
        # 2. Total Bathrooms (IMPORTANT)
        if all(col in X.columns for col in ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']):
            X['TotalBath'] = X['FullBath'] + 0.5*X['HalfBath'] + X['BsmtFullBath'] + 0.5*X['BsmtHalfBath']
        
        # 3. Total Outdoor Space
        outdoor_cols = ['WoodDeckSF', 'OpenPorchSF', 'ScreenPorch']
        available_outdoor = [col for col in outdoor_cols if col in X.columns]
        if available_outdoor:
            X['TotalOutdoorSF'] = X[available_outdoor].sum(axis=1)
        
        # === AGE FEATURES ===
        
        # 4. House Age
        if all(col in X.columns for col in ['YrSold', 'YearBuilt']):
            X['HouseAge'] = X['YrSold'] - X['YearBuilt']
        
        # 5. Remodel Age
        if all(col in X.columns for col in ['YrSold', 'YearRemodAdd', 'YearBuilt']):
            X['RemodAge'] = X['YrSold'] - X['YearRemodAdd']
            X['IsRemodeled'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
        
        # 6. Is New House (built within 5 years)
        if 'HouseAge' in X.columns:
            X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
        
        # === RATIOS (Highly Predictive!) ===
        
        # 7. Living Area Ratio (above grade vs total)
        if 'TotalSF' in X.columns and 'GrLivArea' in X.columns:
            X['LivingAreaRatio'] = X['GrLivArea'] / (X['TotalSF'] + 1)
        
        # 8. Basement Finished Ratio
        if all(col in X.columns for col in ['BsmtFinSF1', 'TotalBsmtSF']):
            X['BsmtFinishedRatio'] = X['BsmtFinSF1'] / (X['TotalBsmtSF'] + 1)
        
        # 9. Lot Area per Room (spaciousness)
        if all(col in X.columns for col in ['LotArea', 'TotRmsAbvGrd']):
            X['LotAreaPerRoom'] = X['LotArea'] / (X['TotRmsAbvGrd'] + 1)
        
        # === QUALITY INTERACTIONS ===
        
        # 10. Quality × Living Area (KEY INTERACTION)
        if all(col in X.columns for col in ['OverallQual', 'GrLivArea']):
            X['QualityArea'] = X['OverallQual'] * X['GrLivArea']
        
        # 11. Overall Score (Quality × Condition)
        if all(col in X.columns for col in ['OverallQual', 'OverallCond']):
            X['OverallScore'] = X['OverallQual'] * X['OverallCond']
        
        # 12. Average Quality Score
        quality_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageQual']
        available_qual = [col for col in quality_cols if col in X.columns]
        if available_qual:
            X['AvgQuality'] = X[available_qual].mean(axis=1)
        
        # 13. Quality Squared (emphasize high quality)
        if 'OverallQual' in X.columns:
            X['QualitySquared'] = X['OverallQual'] ** 2
        
        # === BINARY INDICATORS ===
        
        # 14. Has 2nd Floor
        if '2ndFlrSF' in X.columns:
            X['Has2ndFloor'] = (X['2ndFlrSF'] > 0).astype(int)
        
        # 15. Has Basement
        if 'TotalBsmtSF' in X.columns:
            X['HasBasement'] = (X['TotalBsmtSF'] > 0).astype(int)
        
        # 16. Has Garage
        if 'GarageArea' in X.columns:
            X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
        
        # 17. Has Fireplace
        if 'Fireplaces' in X.columns:
            X['HasFireplace'] = (X['Fireplaces'] > 0).astype(int)
        
        # === GARAGE SCORE ===
        
        # 18. Garage Score (size × quality)
        if all(col in X.columns for col in ['GarageCars', 'GarageQual']):
            X['GarageScore'] = X['GarageCars'] * (X['GarageQual'] + 1)
        
        return X


print("Creating Advanced Features")
print("="*70)

# Apply advanced feature engineering to ordinal-encoded data
adv_engineer = AdvancedFeatureEngineer()
X_train_advanced = adv_engineer.fit_transform(X_train_ordinal)
X_test_advanced = adv_engineer.transform(X_test_ordinal)

print(f"Features before: {X_train_ordinal.shape[1]}")
print(f"Features after: {X_train_advanced.shape[1]}")
print(f"New features created: {X_train_advanced.shape[1] - X_train_ordinal.shape[1]}")

# Apply one-hot encoding
X_train_advanced_full, X_test_advanced_full, _ = apply_onehot_encoding(X_train_advanced, X_test_advanced)

print(f"After one-hot encoding: {X_train_advanced_full.shape[1]} features")

# Add to feature combinations dictionary
feature_combinations['5_AllFeatures+Engineering'] = {
    'X_train': X_train_advanced_full,
    'X_test': X_test_advanced_full,
    'description': 'Complete dataset with engineered features'
}

print("\n✓ Advanced feature engineering complete!")
print("\nUpdated combinations:")
for name, data in feature_combinations.items():
    print(f"  {name}: {data['X_train'].shape[1]} features")

In [ ]:
print("j")

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


### training 

In [ ]:
# Now update the training function to use all combinations

def train_and_evaluate_models():
    """
    Train models on all feature combinations and save results.
    """
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
    import pickle
    from datetime import datetime
    
    # Define models
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(),  # Default alpha=1.0
        'Decision Tree': DecisionTreeRegressor(random_state=42)
    }
    
    results = []
    trained_models = {}
    predictions = {}
    
    print("Training Models on All Combinations")
    print("="*70)
    
    for combo_name, data in feature_combinations.items():
        print(f"\n{combo_name}: {data['description']}")
        print("-"*70)
        
        X_train_combo = data['X_train']
        X_test_combo = data['X_test']
        
        # Ensure no missing values
        X_train_combo = X_train_combo.fillna(0)
        X_test_combo = X_test_combo.fillna(0)
        
        for model_name, model in models.items():
            print(f"  Training {model_name}...", end=" ")
            
            try:
                # Cross-validation
                cv_scores = cross_val_score(
                    model, X_train_combo, y_train_log,
                    cv=5, 
                    scoring='neg_mean_squared_error',
                    n_jobs=-1
                )
                cv_rmse = np.sqrt(-cv_scores.mean())
                cv_std = np.sqrt(cv_scores.std())
                
                # Train on full training set
                model.fit(X_train_combo, y_train_log)
                
                # Training predictions
                y_train_pred = model.predict(X_train_combo)
                train_rmse = np.sqrt(mean_squared_error(y_train_log, y_train_pred))
                train_r2 = r2_score(y_train_log, y_train_pred)
                train_mae = mean_absolute_error(y_train_log, y_train_pred)
                
                # Test predictions
                y_test_pred = model.predict(X_test_combo)
                
                # Store results
                results.append({
                    'Combination': combo_name,
                    'Model': model_name,
                    'CV_RMSE': cv_rmse,
                    'CV_Std': cv_std,
                    'Train_RMSE': train_rmse,
                    'Train_R2': train_r2,
                    'Train_MAE': train_mae,
                    'N_Features': X_train_combo.shape[1]
                })
                
                # Store trained model and predictions
                key = f"{combo_name}_{model_name}"
                trained_models[key] = model
                predictions[key] = {
                    'y_train_true': y_train_log.values,
                    'y_train_pred': y_train_pred,
                    'y_test_pred': y_test_pred,
                    'train_index': X_train_combo.index,
                    'feature_names': X_train_combo.columns.tolist()
                }
                
                print(f"✓ CV RMSE: {cv_rmse:.4f}")
                
            except Exception as e:
                print(f"✗ Error: {str(e)}")
                continue
    
    # Save results
    results_df = pd.DataFrame(results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    with open(f'model_results_{timestamp}.pkl', 'wb') as f:
        pickle.dump({
            'results_df': results_df,
            'trained_models': trained_models,
            'predictions': predictions,
            'timestamp': timestamp
        }, f)
    
    print("\n" + "="*70)
    print(f"✓ Training complete! Results saved to: model_results_{timestamp}.pkl")
    print(f"  Total combinations: {len(feature_combinations)}")
    print(f"  Models per combination: {len(models)}")
    print(f"  Total trained: {len(trained_models)}")
    
    return results_df, trained_models, predictions, timestamp


# Run training
results_df, trained_models, predictions, timestamp = train_and_evaluate_models()

## Results summary

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def load_and_summarize_results(filename):
    """
    Load saved results and create comprehensive summary.
    """
    # Load results
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    
    results_df = data['results_df']
    trained_models = data['trained_models']
    predictions = data['predictions']
    
    print("Model Performance Summary")
    print("="*70)
    
    # 1. Overall Summary Table
    print("\n1. COMPLETE RESULTS:")
    print("-"*70)
    summary = results_df.copy()
    summary = summary.sort_values('CV_RMSE')
    print(summary.to_string(index=False))
    
    # 2. Best Model per Combination
    print("\n\n2. BEST MODEL FOR EACH COMBINATION:")
    print("-"*70)
    best_per_combo = results_df.loc[results_df.groupby('Combination')['CV_RMSE'].idxmin()]
    print(best_per_combo[['Combination', 'Model', 'CV_RMSE', 'Train_R2']].to_string(index=False))
    
    # 3. Best Model Overall
    print("\n\n3. OVERALL BEST MODEL:")
    print("-"*70)
    best_overall = results_df.loc[results_df['CV_RMSE'].idxmin()]
    print(f"Combination: {best_overall['Combination']}")
    print(f"Model: {best_overall['Model']}")
    print(f"CV RMSE: {best_overall['CV_RMSE']:.4f} (±{best_overall['CV_Std']:.4f})")
    print(f"Train R²: {best_overall['Train_R2']:.4f}")
    print(f"Features: {best_overall['N_Features']}")
    
    # 4. Visualizations
    create_performance_visualizations(results_df)
    
    return results_df, trained_models, predictions


def create_performance_visualizations(results_df):
    """
    Create comprehensive visualizations of model performance.
    """
    fig = plt.figure(figsize=(16, 12))
    
    # 1. CV RMSE Comparison
    ax1 = plt.subplot(2, 3, 1)
    pivot_cv = results_df.pivot(index='Model', columns='Combination', values='CV_RMSE')
    pivot_cv.plot(kind='bar', ax=ax1, width=0.8)
    ax1.set_title('Cross-Validation RMSE by Model and Combination', fontweight='bold')
    ax1.set_ylabel('CV RMSE (log scale)')
    ax1.set_xlabel('Model')
    ax1.legend(title='Combination', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(axis='y', alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 2. Train R² Comparison
    ax2 = plt.subplot(2, 3, 2)
    pivot_r2 = results_df.pivot(index='Model', columns='Combination', values='Train_R2')
    pivot_r2.plot(kind='bar', ax=ax2, width=0.8)
    ax2.set_title('Training R² by Model and Combination', fontweight='bold')
    ax2.set_ylabel('R² Score')
    ax2.set_xlabel('Model')
    ax2.legend(title='Combination', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(axis='y', alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 3. CV RMSE with Error Bars
    ax3 = plt.subplot(2, 3, 3)
    for combo in results_df['Combination'].unique():
        data = results_df[results_df['Combination'] == combo]
        x_pos = np.arange(len(data))
        ax3.errorbar(x_pos, data['CV_RMSE'], yerr=data['CV_Std'], 
                     marker='o', capsize=5, label=combo, linewidth=2)
    ax3.set_xticks(range(len(results_df['Model'].unique())))
    ax3.set_xticklabels(results_df['Model'].unique(), rotation=45, ha='right')
    ax3.set_title('CV RMSE with Standard Deviation', fontweight='bold')
    ax3.set_ylabel('CV RMSE (log scale)')
    ax3.legend(title='Combination')
    ax3.grid(alpha=0.3)
    
    # 4. Overfitting Analysis (Train vs CV RMSE)
    ax4 = plt.subplot(2, 3, 4)
    for model in results_df['Model'].unique():
        data = results_df[results_df['Model'] == model]
        ax4.scatter(data['Train_RMSE'], data['CV_RMSE'], 
                   s=100, alpha=0.6, label=model)
    # Add diagonal line (perfect match)
    max_val = max(results_df['Train_RMSE'].max(), results_df['CV_RMSE'].max())
    ax4.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Perfect Match')
    ax4.set_xlabel('Training RMSE')
    ax4.set_ylabel('CV RMSE')
    ax4.set_title('Overfitting Analysis', fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    # 5. Model Ranking
    ax5 = plt.subplot(2, 3, 5)
    top_10 = results_df.nsmallest(10, 'CV_RMSE')
    y_pos = np.arange(len(top_10))
    colors = ['green' if i < 3 else 'steelblue' for i in range(len(top_10))]
    ax5.barh(y_pos, top_10['CV_RMSE'], color=colors, alpha=0.7)
    ax5.set_yticks(y_pos)
    ax5.set_yticklabels([f"{row['Model']}\n({row['Combination'][:20]}...)" 
                          for _, row in top_10.iterrows()], fontsize=8)
    ax5.set_xlabel('CV RMSE (log scale)')
    ax5.set_title('Top 10 Model-Combination Pairs', fontweight='bold')
    ax5.invert_yaxis()
    ax5.grid(axis='x', alpha=0.3)
    
    # 6. Feature Count Impact
    ax6 = plt.subplot(2, 3, 6)
    for model in results_df['Model'].unique():
        data = results_df[results_df['Model'] == model]
        ax6.scatter(data['N_Features'], data['CV_RMSE'], 
                   s=100, alpha=0.6, label=model)
    ax6.set_xlabel('Number of Features')
    ax6.set_ylabel('CV RMSE (log scale)')
    ax6.set_title('Feature Count vs Performance', fontweight='bold')
    ax6.legend()
    ax6.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualizations created!")


# Usage:
# Replace with your actual filename
results_df, trained_models, predictions = load_and_summarize_results(f'model_results_{timestamp}.pkl')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def visualize_competition_metric(results_df):
    """
    Visualize model performance using Kaggle competition metric (RMSE on log scale).
    Lower is better.
    """
    # Sort by CV RMSE (competition metric)
    results_sorted = results_df.sort_values('CV_RMSE')
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create color map for models
    model_colors = {
        'Linear Regression': '#3498db',
        'Ridge Regression': '#2ecc71',
        'Decision Tree': '#e74c3c'
    }
    
    # Plot bars
    y_pos = np.arange(len(results_sorted))
    colors = [model_colors[model] for model in results_sorted['Model']]
    
    bars = ax.barh(y_pos, results_sorted['CV_RMSE'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add error bars (CV standard deviation)
    ax.errorbar(results_sorted['CV_RMSE'], y_pos, 
                xerr=results_sorted['CV_Std'], 
                fmt='none', ecolor='black', capsize=5, capthick=2, alpha=0.6)
    
    # Customize y-axis labels
    labels = []
    for _, row in results_sorted.iterrows():
        combo_short = row['Combination'].replace('_', ' ')
        label = f"{row['Model']}\n({combo_short})"
        labels.append(label)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    
    # Highlight top 3
    for i in range(min(3, len(bars))):
        bars[i].set_edgecolor('gold')
        bars[i].set_linewidth(3)
    
    # Labels and title
    ax.set_xlabel('Cross-Validation RMSE (log scale)', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance - Kaggle Competition Metric (Lower is Better)', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, edgecolor='black', label=model) 
                      for model, color in model_colors.items()]
    legend_elements.append(Patch(facecolor='white', edgecolor='gold', linewidth=3, label='Top 3'))
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    
    # Add values on bars
    for i, (idx, row) in enumerate(results_sorted.iterrows()):
        ax.text(row['CV_RMSE'] + 0.002, i, f"{row['CV_RMSE']:.4f}", 
                va='center', fontsize=8, fontweight='bold')
    
    # Invert y-axis so best is on top
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("="*70)
    print("COMPETITION METRIC RANKING (CV RMSE):")
    print("="*70)
    for i, (_, row) in enumerate(results_sorted.iterrows(), 1):
        print(f"{i:2d}. {row['Model']:20s} | {row['Combination']:30s} | "
              f"RMSE: {row['CV_RMSE']:.4f} (±{row['CV_Std']:.4f})")
    
    print(f"\n🏆 BEST MODEL: {results_sorted.iloc[0]['Model']} "
          f"with {results_sorted.iloc[0]['Combination']}")
    print(f"   CV RMSE: {results_sorted.iloc[0]['CV_RMSE']:.4f}")


# Usage:
visualize_competition_metric(results_df)

In [ ]:
# Get best model
best_idx = results_df['CV_RMSE'].idxmin()
best_model_info = results_df.loc[best_idx]
best_key = f"{best_model_info['Combination']}_{best_model_info['Model']}"

# Get predictions
pred_data = predictions[best_key]
y_true_log = pred_data['y_train_true']
y_pred_log = pred_data['y_train_pred']

# Convert back to dollars
y_true_dollars = np.expm1(y_true_log)
y_pred_dollars = np.expm1(y_pred_log)
errors_dollars = y_true_dollars - y_pred_dollars

# Display errors
print(f"Best Model: {best_model_info['Model']} - {best_model_info['Combination']}")
print("="*70)
print(f"Mean Absolute Error: ${np.abs(errors_dollars).mean():,.0f}")
print(f"Median Absolute Error: ${np.median(np.abs(errors_dollars)):,.0f}")
print(f"Root Mean Squared Error: ${np.sqrt((errors_dollars**2).mean()):,.0f}")
print(f"\nMax Overestimate: ${errors_dollars.min():,.0f}")
print(f"Max Underestimate: ${errors_dollars.max():,.0f}")

### error analyse

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_predict

def analyze_model_errors(predictions, results_df, trained_models, feature_combinations, y_train_log):
    """
    Analyze the types of errors each model makes using cross-validation predictions.
    """
    print("Error Analysis: Types of Errors Models Make (Cross-Validation)")
    print("="*70)
    
    # Get best model from each type
    best_models = {}
    for model_type in results_df['Model'].unique():
        best = results_df[results_df['Model'] == model_type].nsmallest(1, 'CV_RMSE').iloc[0]
        key = f"{best['Combination']}_{best['Model']}"
        best_models[model_type] = {
            'key': key,
            'combination': best['Combination']
        }
    
    fig = plt.figure(figsize=(18, 12))
    
    for idx, (model_type, info) in enumerate(best_models.items(), 1):
        key = info['key']
        combo_name = info['combination']
        
        # Get the model and data
        model = trained_models[key]
        X_train_combo = feature_combinations[combo_name]['X_train'].fillna(0)
        
        # Get cross-validation predictions (out-of-sample!)
        print(f"\nGenerating CV predictions for {model_type}...", end=" ")
        y_pred_cv = cross_val_predict(model, X_train_combo, y_train_log, cv=5)
        print("✓")
        
        y_true = y_train_log.values
        
        # Calculate errors (on log scale)
        errors = y_true - y_pred_cv
        abs_errors = np.abs(errors)
        
        # Convert back to original scale for interpretation
        y_true_orig = np.expm1(y_true)
        y_pred_orig = np.expm1(y_pred_cv)
        errors_orig = y_true_orig - y_pred_orig
        
        print(f"\n{model_type}:")
        print("-"*70)
        print(f"Mean Absolute Error (log): {abs_errors.mean():.4f}")
        print(f"Mean Absolute Error ($): ${np.abs(errors_orig).mean():,.0f}")
        print(f"Median Absolute Error ($): ${np.median(np.abs(errors_orig)):,.0f}")
        print(f"RMSE ($): ${np.sqrt((errors_orig**2).mean()):,.0f}")
        print(f"Max Overestimate ($): ${errors_orig.min():,.0f}")
        print(f"Max Underestimate ($): ${errors_orig.max():,.0f}")
        
        # 1. Residual Plot
        ax1 = plt.subplot(3, 4, idx)
        ax1.scatter(y_pred_cv, errors, alpha=0.5, s=20)
        ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
        ax1.set_xlabel('Predicted (log)')
        ax1.set_ylabel('Residuals (log)')
        ax1.set_title(f'{model_type}\nResidual Plot (CV)', fontweight='bold')
        ax1.grid(alpha=0.3)
        
        # 2. Residual Distribution
        ax2 = plt.subplot(3, 4, idx + 4)
        ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7)
        ax2.axvline(x=0, color='r', linestyle='--', linewidth=2)
        ax2.set_xlabel('Residuals (log)')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Residual Distribution', fontweight='bold')
        ax2.grid(alpha=0.3)
        
        # 3. Actual vs Predicted
        ax3 = plt.subplot(3, 4, idx + 8)
        ax3.scatter(y_true_orig/1000, y_pred_orig/1000, alpha=0.5, s=20)
        max_val = max(y_true_orig.max(), y_pred_orig.max()) / 1000
        ax3.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
        ax3.set_xlabel('Actual Price ($1000s)')
        ax3.set_ylabel('Predicted Price ($1000s)')
        ax3.set_title('Actual vs Predicted', fontweight='bold')
        ax3.legend()
        ax3.grid(alpha=0.3)
        
        # Identify error patterns
        print(f"\nError Patterns:")
        
        # Overestimation vs underestimation
        overestimate = (errors < 0).sum()
        underestimate = (errors > 0).sum()
        print(f"  - Overestimates: {overestimate} ({overestimate/len(errors)*100:.1f}%)")
        print(f"  - Underestimates: {underestimate} ({underestimate/len(errors)*100:.1f}%)")
        
        # Large errors
        large_errors = (abs_errors > np.quantile(abs_errors, 0.95)).sum()
        print(f"  - Large errors (>95th percentile): {large_errors} ({large_errors/len(errors)*100:.1f}%)")
        
        # Errors by price range
        try:
            price_bins = pd.qcut(y_true_orig, q=4, labels=['Low', 'Mid-Low', 'Mid-High', 'High'], duplicates='drop')
            print(f"\n  Mean Absolute Error by Price Range:")
            for price_range in ['Low', 'Mid-Low', 'Mid-High', 'High']:
                if price_range in price_bins.values:
                    mask = price_bins == price_range
                    mae = np.abs(errors_orig[mask]).mean()
                    print(f"    {price_range}: ${mae:,.0f}")
        except Exception as e:
            print(f"  Could not compute errors by price range: {e}")
    
    plt.tight_layout()
    plt.show()
    
    # Additional analysis: Compare error patterns across models
    print("\n" + "="*70)
    print("COMPARATIVE ERROR ANALYSIS:")
    print("="*70)
    
    fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    error_comparison = []
    for model_type, info in best_models.items():
        key = info['key']
        combo_name = info['combination']
        
        model = trained_models[key]
        X_train_combo = feature_combinations[combo_name]['X_train'].fillna(0)
        
        # Get CV predictions
        y_pred_cv = cross_val_predict(model, X_train_combo, y_train_log, cv=5)
        errors = y_train_log.values - y_pred_cv
        
        error_comparison.append({
            'Model': model_type,
            'errors': errors
        })
    
    # 1. Error distribution comparison
    for item in error_comparison:
        axes[0].hist(item['errors'], bins=50, alpha=0.5, label=item['Model'], edgecolor='black')
    axes[0].axvline(x=0, color='r', linestyle='--', linewidth=2)
    axes[0].set_xlabel('Residuals (log)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Error Distribution Comparison', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # 2. Box plot comparison
    error_data = pd.DataFrame({item['Model']: item['errors'] for item in error_comparison})
    error_data.boxplot(ax=axes[1])
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_ylabel('Residuals (log)')
    axes[1].set_title('Error Range Comparison', fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    # 3. Absolute error comparison
    abs_error_means = [np.abs(item['errors']).mean() for item in error_comparison]
    model_names = [item['Model'] for item in error_comparison]
    axes[2].bar(model_names, abs_error_means, alpha=0.7, edgecolor='black')
    axes[2].set_ylabel('Mean Absolute Error (log)')
    axes[2].set_title('Mean Absolute Error Comparison', fontweight='bold')
    axes[2].grid(axis='y', alpha=0.3)
    plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Error analysis complete!")


# Usage (with additional parameters):
analyze_model_errors(predictions, results_df, trained_models, feature_combinations, y_train_log)

In [ ]:
# Find best Ridge model
best_ridge = results_df[results_df['Model'] == 'Ridge Regression'].nsmallest(1, 'CV_RMSE').iloc[0]
best_ridge_key = f"{best_ridge['Combination']}_{best_ridge['Model']}"

print(f"Best Ridge: {best_ridge['Combination']} - RMSE: {best_ridge['CV_RMSE']:.4f}")

### ensembel

In [ ]:
# Get best Ridge model
best_ridge_key = f"{best_ridge['Combination']}_{best_ridge['Model']}"
model = trained_models[best_ridge_key]
X_data = feature_combinations[best_ridge['Combination']]['X_train']

# Get coefficients
feature_importance = pd.DataFrame({
    'Feature': X_data.columns,
    'Coefficient': np.abs(model.coef_)  # Absolute value
}).sort_values('Coefficient', ascending=False)

print("Top 20 Most Important Features (Ridge):")
print(feature_importance.head(20).to_string(index=False))

# Visualize
plt.figure(figsize=(10, 8))
plt.barh(range(20), feature_importance.head(20)['Coefficient'])
plt.yticks(range(20), feature_importance.head(20)['Feature'])
plt.xlabel('Absolute Coefficient')
plt.title('Top 20 Feature Importances (Ridge)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Feature importance 

In [ ]:
# Get Decision Tree from combination 5
best_dt_combo5 = results_df[
    (results_df['Model'] == 'Decision Tree') & 
    (results_df['Combination'] == '5_AllFeatures+Engineering')
].iloc[0]

print(f"Decision Tree - Combination 5: RMSE = {best_dt_combo5['CV_RMSE']:.4f}")

# Get model and data
dt_key_combo5 = f"{best_dt_combo5['Combination']}_{best_dt_combo5['Model']}"
dt_model_combo5 = trained_models[dt_key_combo5]
X_dt_combo5 = feature_combinations['5_AllFeatures+Engineering']['X_train'].fillna(0)

print(f"\nFeatures: {X_dt_combo5.shape[1]}")
print(f"Model importances: {len(dt_model_combo5.feature_importances_)}")

# Check if they match
if X_dt_combo5.shape[1] == len(dt_model_combo5.feature_importances_):
    feature_importance_dt5 = pd.DataFrame({
        'Feature': X_dt_combo5.columns,
        'Importance': dt_model_combo5.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 20 Most Important Features (Decision Tree - Combo 5):")
    print(feature_importance_dt5.head(20).to_string(index=False))
    
    # Visualize
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 8))
    top_20 = feature_importance_dt5.head(20)
    plt.barh(range(len(top_20)), top_20['Importance'])
    plt.yticks(range(len(top_20)), top_20['Feature'])
    plt.xlabel('Importance')
    plt.title('Top 20 Feature Importances - Decision Tree (Combo 5)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️ Mismatch! Re-run training.")

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

print("Model Ensembling")
print("="*70)

# Get best models from each type
best_lr = results_df[results_df['Model'] == 'Linear Regression'].nsmallest(1, 'CV_RMSE').iloc[0]
best_ridge = results_df[results_df['Model'] == 'Ridge Regression'].nsmallest(1, 'CV_RMSE').iloc[0]
best_dt = results_df[results_df['Model'] == 'Decision Tree'].nsmallest(1, 'CV_RMSE').iloc[0]

# Get their CV predictions
lr_key = f"{best_lr['Combination']}_{best_lr['Model']}"
ridge_key = f"{best_ridge['Combination']}_{best_ridge['Model']}"
dt_key = f"{best_dt['Combination']}_{best_dt['Model']}"

X_lr = feature_combinations[best_lr['Combination']]['X_train'].fillna(0)
X_ridge = feature_combinations[best_ridge['Combination']]['X_train'].fillna(0)
X_dt = feature_combinations[best_dt['Combination']]['X_train'].fillna(0)

from sklearn.model_selection import cross_val_predict

lr_pred = cross_val_predict(trained_models[lr_key], X_lr, y_train_log, cv=5)
ridge_pred = cross_val_predict(trained_models[ridge_key], X_ridge, y_train_log, cv=5)
dt_pred = cross_val_predict(trained_models[dt_key], X_dt, y_train_log, cv=5)

# Ensemble combinations
ensembles = {
    'Ridge + Linear (avg)': (ridge_pred + lr_pred) / 2,
    'Ridge + DT (avg)': (ridge_pred + dt_pred) / 2,
    'All 3 (avg)': (ridge_pred + lr_pred + dt_pred) / 3,
    'Ridge (70%) + Linear (30%)': 0.7*ridge_pred + 0.3*lr_pred,
    'Ridge (80%) + DT (20%)': 0.8*ridge_pred + 0.2*dt_pred,
}

# Calculate RMSE for each
results = []
for name, preds in ensembles.items():
    rmse = np.sqrt(mean_squared_error(y_train_log, preds))
    results.append({'Ensemble': name, 'CV_RMSE': rmse})

results_df_ensemble = pd.DataFrame(results).sort_values('CV_RMSE')

# Add individual models for comparison
results_df_ensemble = pd.concat([
    pd.DataFrame([
        {'Ensemble': 'Ridge (best single)', 'CV_RMSE': best_ridge['CV_RMSE']},
        {'Ensemble': 'Linear (best single)', 'CV_RMSE': best_lr['CV_RMSE']},
        {'Ensemble': 'Decision Tree', 'CV_RMSE': best_dt['CV_RMSE']}
    ]),
    results_df_ensemble
]).sort_values('CV_RMSE').reset_index(drop=True)

print("\nEnsemble Results:")
print(results_df_ensemble.to_string(index=False))

best_ensemble = results_df_ensemble.iloc[0]
print(f"\n🏆 Best: {best_ensemble['Ensemble']} - RMSE: {best_ensemble['CV_RMSE']:.4f}")

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


## RandomizedSearchCV on Linear Model

In [ ]:
if True:
    from sklearn.linear_model import ElasticNet
    from sklearn.model_selection import RandomizedSearchCV
    from scipy.stats import uniform, loguniform
    import numpy as np
    import pandas as pd
    
    print("Comprehensive Linear Model Tuning - ElasticNet")
    print("="*70)
    
    # Use best combination
    best_combo = '5_AllFeatures+Engineering'
    X_train_tuning = feature_combinations[best_combo]['X_train'].fillna(0)
    
    print(f"Training data shape: {X_train_tuning.shape}")
    print(f"Number of features: {X_train_tuning.shape[1]}")
    
    # Extended parameter distributions for ElasticNet
    param_distributions = {
        'alpha': loguniform(0.0001, 100),        # Regularization strength (wide range)
        'l1_ratio': uniform(0, 1),                # 0=Ridge, 1=Lasso, 0.5=Equal mix
        'fit_intercept': [True, False],
        'max_iter': [5000, 10000, 20000],
        'tol': loguniform(1e-6, 1e-2),
        'selection': ['cyclic', 'random']
    }
    
    print(f"\nRandomized Search Configuration:")
    print(f"  Parameters to tune: {list(param_distributions.keys())}")
    print(f"  Iterations: 500")
    print(f"  Cross-validation folds: 5")
    print(f"  Total fits: 2,500")
    
    print("\nRunning Randomized Search (this may take 5-10 minutes)...")
    print("-"*70)
    
    random_search = RandomizedSearchCV(
        estimator=ElasticNet(random_state=42),
        param_distributions=param_distributions,
        n_iter=500,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_train_tuning, y_train_log)
    
    best_rmse = np.sqrt(-random_search.best_score_)
    
    print(f"\n✓ Best ElasticNet RMSE: {best_rmse:.4f}")
    print(f"\nBest parameters:")
    for param, value in random_search.best_params_.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.6f}")
        else:
            print(f"  {param}: {value}")
    
    # Interpret l1_ratio
    l1_ratio = random_search.best_params_['l1_ratio']
    if l1_ratio < 0.1:
        model_type = "Almost pure Ridge (L2)"
    elif l1_ratio > 0.9:
        model_type = "Almost pure Lasso (L1)"
    elif 0.4 <= l1_ratio <= 0.6:
        model_type = "Balanced ElasticNet (L1+L2)"
    else:
        model_type = f"ElasticNet ({int(l1_ratio*100)}% L1, {int((1-l1_ratio)*100)}% L2)"
    
    print(f"\nModel type: {model_type}")
    
    # Summary comparison
    print("\n" + "="*70)
    print("LINEAR MODEL COMPARISON:")
    print("="*70)
    
    # Compare with previous linear models
    baseline_ridge = results_df[
        (results_df['Model'] == 'Ridge Regression') & 
        (results_df['Combination'] == best_combo)
    ]['CV_RMSE'].values[0]
    
    baseline_linear = results_df[
        (results_df['Model'] == 'Linear Regression') & 
        (results_df['Combination'] == best_combo)
    ]['CV_RMSE'].values[0]
    
    # Tuned Ridge from earlier (if available)
    tuned_ridge_rmse = 0.1111  # From your previous tuning
    
    comparison = pd.DataFrame([
        {'Model': 'Linear Regression (baseline)', 'CV_RMSE': baseline_linear},
        {'Model': 'Ridge (baseline, alpha=1.0)', 'CV_RMSE': baseline_ridge},
        {'Model': 'Ridge (tuned)', 'CV_RMSE': tuned_ridge_rmse},
        {'Model': 'ElasticNet (tuned)', 'CV_RMSE': best_rmse}
    ]).sort_values('CV_RMSE')
    
    print(comparison.to_string(index=False))
    
    # Best linear model overall
    best_linear = comparison.iloc[0]
    print(f"\n🏆 BEST LINEAR MODEL: {best_linear['Model']}")
    print(f"   CV RMSE: {best_linear['CV_RMSE']:.4f}")
    
    # Calculate improvement over baseline Ridge
    improvement = (baseline_ridge - best_rmse) / baseline_ridge * 100
    print(f"\n📊 ElasticNet vs Baseline Ridge:")
    print(f"   Baseline Ridge: {baseline_ridge:.4f}")
    print(f"   Tuned ElasticNet: {best_rmse:.4f}")
    if improvement > 0:
        print(f"   Improvement: {improvement:.2f}%")
    else:
        print(f"   Ridge still better by {-improvement:.2f}%")
    
    # Visualize
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(10, 6))
    colors = ['green' if i == 0 else 'steelblue' for i in range(len(comparison))]
    bars = plt.barh(range(len(comparison)), comparison['CV_RMSE'], color=colors)
    plt.yticks(range(len(comparison)), comparison['Model'])
    plt.xlabel('CV RMSE (log scale)')
    plt.title('Linear Model Performance Comparison', fontweight='bold')
    plt.axvline(x=best_linear['CV_RMSE'], color='gold', linestyle='--', linewidth=2, alpha=0.7)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    
    # Add values on bars
    for i, (idx, row) in enumerate(comparison.iterrows()):
        plt.text(row['CV_RMSE'] + 0.001, i, f"{row['CV_RMSE']:.4f}", 
                 va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Save best model
    best_elasticnet_model = random_search.best_estimator_
    
    print("\n✓ Best linear model tuning complete!")
    print(f"✓ Best ElasticNet model saved")
    
    # Show top 10 parameter combinations
    print("\nTop 10 parameter combinations tried:")
    results = pd.DataFrame(random_search.cv_results_)
    top_10 = results.nsmallest(10, 'mean_test_score')[
        ['param_alpha', 'param_l1_ratio', 'mean_test_score', 'std_test_score']
    ].copy()
    top_10['CV_RMSE'] = np.sqrt(-top_10['mean_test_score'])
    top_10 = top_10.drop('mean_test_score', axis=1)
    print(top_10.to_string(index=False))

## RandomizedSearchCV on ridge

In [ ]:
if False:    
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.linear_model import Ridge
    from scipy.stats import loguniform
    import numpy as np
    
    print("Intensive Hyperparameter Tuning - Ridge Regression")
    print("="*70)
    
    # Use best combination
    best_combo = '5_AllFeatures+Engineering'
    X_train_tuning = feature_combinations[best_combo]['X_train'].fillna(0)
    
    print(f"Training data shape: {X_train_tuning.shape}")
    print(f"Number of features: {X_train_tuning.shape[1]}")
    
    print("\nRandomized Search (500 iterations)")
    print("-"*70)
    
    # Extended parameter distributions
    params = {
        'alpha': loguniform(0.001, 1000),  # Very wide range in log space
        'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'saga'],
        'fit_intercept': [True, False],
        'tol': loguniform(1e-6, 1e-1),
        'max_iter': [5000]  # Add max iterations
    }
    
    random_search = RandomizedSearchCV(
        estimator=Ridge(random_state=42),
        param_distributions=params,
        n_iter=500,  # 500 random combinations
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_train_tuning, y_train_log)
    
    best_rmse = np.sqrt(-random_search.best_score_)
    
    print(f"\n✓ Best RMSE: {best_rmse:.4f}")
    print(f"Best parameters:")
    for param, value in random_search.best_params_.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.6f}")
        else:
            print(f"  {param}: {value}")
    
    # Summary
    print("\n" + "="*70)
    print("TUNING RESULTS SUMMARY:")
    print("="*70)
    
    baseline_ridge = results_df[
        (results_df['Model'] == 'Ridge Regression') & 
        (results_df['Combination'] == best_combo)
    ]['CV_RMSE'].values[0]
    
    comparison = pd.DataFrame([
        {'Model': 'Ridge (baseline, alpha=1.0)', 'CV_RMSE': baseline_ridge},
        {'Model': 'Ridge (tuned, 500 iterations)', 'CV_RMSE': best_rmse}
    ]).sort_values('CV_RMSE')
    
    print(comparison.to_string(index=False))
    
    # Calculate improvement
    improvement = (baseline_ridge - best_rmse) / baseline_ridge * 100
    
    print(f"\n📊 Ridge Improvement:")
    print(f"  Baseline: {baseline_ridge:.4f}")
    print(f"  After Tuning: {best_rmse:.4f}")
    print(f"  Improvement: {improvement:.2f}%")
    
    print(f"\n🏆 Best Model: Ridge (tuned) - RMSE: {best_rmse:.4f}")
    print(f"\nBest alpha: {random_search.best_params_['alpha']:.6f}")
    print(f"This is {random_search.best_params_['alpha']/1.0:.2f}x the default alpha=1.0")
    
    # Save best tuned model
    best_tuned_model = random_search.best_estimator_

In [ ]:
if False:
# Print and save best Ridge parameters
    print("\n" + "="*70)
    print("BEST RIDGE MODEL - FINAL PARAMETERS")
    print("="*70)
    
    best_params = random_search.best_params_
    best_score = np.sqrt(-random_search.best_score_)
    
    print(f"\nCross-Validation RMSE: {best_score:.4f}")
    print(f"\nOptimal Hyperparameters:")
    for param, value in best_params.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.6f}")
        else:
            print(f"  {param}: {value}")
    
    # Save to file
    import json
    save_dict = {
        'model': 'Ridge Regression',
        'combination': best_combo,
        'cv_rmse': float(best_score),
        'parameters': {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                       for k, v in best_params.items()}
    }
    
    with open('best_ridge_params.json', 'w') as f:
        json.dump(save_dict, f, indent=4)
    
    print(f"\n✓ Parameters saved to: best_ridge_params.json")


## Decision tree gridsearch

In [ ]:
if False:
    from sklearn.model_selection import GridSearchCV
    from sklearn.tree import DecisionTreeRegressor
    import numpy as np
    
    print("Hyperparameter Tuning - Decision Tree")
    print("="*70)
    
    # Find best Decision Tree combination
    best_dt = results_df[results_df['Model'] == 'Decision Tree'].nsmallest(1, 'CV_RMSE').iloc[0]
    best_combo = best_dt['Combination']
    
    print(f"Best Decision Tree baseline:")
    print(f"  Combination: {best_combo}")
    print(f"  CV RMSE: {best_dt['CV_RMSE']:.4f}")
    
    # Get data
    X_train_tuning = feature_combinations[best_combo]['X_train'].fillna(0)
    
    print(f"\nTraining data shape: {X_train_tuning.shape}")
    print(f"Number of features: {X_train_tuning.shape[1]}")
    
    # Define comprehensive parameter grid
    
    param_grid = {
        'max_depth': [5, 10, 15, 20, None],           # 8→5
        'min_samples_split': [2, 10, 50],              # 5→3
        'min_samples_leaf': [1, 4, 16],                # 5→3
        'max_features': ['sqrt','log2', None],               
        'criterion': ['squared_error', 'friedman_mse'] # 3→2
    }
    print(f"\nGrid Search Configuration:")
    print(f"  Total combinations: {np.prod([len(v) for v in param_grid.values()])}")
    print(f"  This will test all combinations with 5-fold CV")
    print(f"  Estimated time: several minutes...")
    
    print("\nRunning Grid Search...")
    print("-"*70)
    
    grid_search = GridSearchCV(
        estimator=DecisionTreeRegressor(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train_tuning, y_train_log)
    
    best_rmse = np.sqrt(-grid_search.best_score_)
    
    print(f"\n✓ Best RMSE: {best_rmse:.4f}")
    print(f"\nBest parameters:")
    for param, value in grid_search.best_params_.items():
        print(f"  {param}: {value}")
    
    # Summary
    print("\n" + "="*70)
    print("TUNING RESULTS SUMMARY:")
    print("="*70)
    
    comparison = pd.DataFrame([
        {'Model': 'Decision Tree (baseline)', 'CV_RMSE': best_dt['CV_RMSE']},
        {'Model': 'Decision Tree (tuned)', 'CV_RMSE': best_rmse}
    ]).sort_values('CV_RMSE')
    
    print(comparison.to_string(index=False))
    
    # Calculate improvement
    improvement = (best_dt['CV_RMSE'] - best_rmse) / best_dt['CV_RMSE'] * 100
    
    print(f"\n📊 Decision Tree Improvement:")
    print(f"  Baseline: {best_dt['CV_RMSE']:.4f}")
    print(f"  After Tuning: {best_rmse:.4f}")
    print(f"  Improvement: {improvement:.2f}%")
    
    # Compare with best Ridge
    best_ridge = results_df[results_df['Model'] == 'Ridge Regression'].nsmallest(1, 'CV_RMSE').iloc[0]
    print(f"\n📊 Comparison with Ridge:")
    print(f"  Best Ridge: {best_ridge['CV_RMSE']:.4f}")
    print(f"  Tuned Decision Tree: {best_rmse:.4f}")
    
    if best_rmse < best_ridge['CV_RMSE']:
        print(f"  🏆 Decision Tree wins by {(best_ridge['CV_RMSE'] - best_rmse):.4f}!")
    else:
        print(f"  Ridge still better by {(best_rmse - best_ridge['CV_RMSE']):.4f}")
    
    # Save best tuned model
    best_tuned_dt = grid_search.best_estimator_
    print(f"\n✓ Best Decision Tree model saved")

## Train with fine tuned hyperparameters

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

print("Training Optimized Models (with Best ElasticNet)")
print("="*70)

# Use best combination
best_combo = '5_AllFeatures+Engineering'
X_train_final = feature_combinations[best_combo]['X_train'].fillna(0)

# 1. Train optimized ElasticNet (BEST LINEAR MODEL)
print("\n1. Training ElasticNet with optimized parameters...")
elasticnet_optimized = ElasticNet(
    alpha=0.000277,
    l1_ratio=0.695411,
    fit_intercept=True,
    max_iter=5000,
    tol=0.000005,
    selection='random',
    random_state=42
)
elasticnet_optimized.fit(X_train_final, y_train_log)
elasticnet_pred_cv = cross_val_predict(elasticnet_optimized, X_train_final, y_train_log, cv=5)
elasticnet_rmse = np.sqrt(mean_squared_error(y_train_log, elasticnet_pred_cv))
print(f"✓ ElasticNet CV RMSE: {elasticnet_rmse:.4f}")

# 2. Train optimized Decision Tree
print("\n2. Training Decision Tree with optimized parameters...")
dt_optimized = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=10,
    max_features=None,
    min_samples_leaf=4,
    min_samples_split=2,
    random_state=42
)
dt_optimized.fit(X_train_final, y_train_log)
dt_pred_cv = cross_val_predict(dt_optimized, X_train_final, y_train_log, cv=5)
dt_rmse = np.sqrt(mean_squared_error(y_train_log, dt_pred_cv))
print(f"✓ Decision Tree CV RMSE: {dt_rmse:.4f}")

# 3. Create Ensembles
print("\n" + "="*70)
print("Creating Ensembles")
print("="*70)

ensembles = {
    'ElasticNet (optimized)': elasticnet_pred_cv,
    'Decision Tree (optimized)': dt_pred_cv,
    'Simple Average (ElasticNet + DT)': (elasticnet_pred_cv + dt_pred_cv) / 2,
    'Weighted 70/30 (ElasticNet/DT)': 0.7 * elasticnet_pred_cv + 0.3 * dt_pred_cv,
    'Weighted 80/20 (ElasticNet/DT)': 0.8 * elasticnet_pred_cv + 0.2 * dt_pred_cv,
    'Weighted 90/10 (ElasticNet/DT)': 0.9 * elasticnet_pred_cv + 0.1 * dt_pred_cv,
    'Weighted 95/5 (ElasticNet/DT)': 0.95 * elasticnet_pred_cv + 0.05 * dt_pred_cv,
    'Weighted 60/40 (ElasticNet/DT)': 0.6 * elasticnet_pred_cv + 0.4 * dt_pred_cv,
}

# Calculate RMSE for each ensemble
results = []
for name, predictions in ensembles.items():
    rmse = np.sqrt(mean_squared_error(y_train_log, predictions))
    results.append({
        'Model/Ensemble': name,
        'CV_RMSE': rmse
    })

results_df_final = pd.DataFrame(results).sort_values('CV_RMSE')

print("\nEnsemble Results:")
print(results_df_final.to_string(index=False))

# Highlight best
best_model = results_df_final.iloc[0]
print(f"\n🏆 BEST MODEL: {best_model['Model/Ensemble']}")
print(f"   CV RMSE: {best_model['CV_RMSE']:.4f}")

# Compare with baseline
baseline_ridge = results_df[
    (results_df['Model'] == 'Ridge Regression') & 
    (results_df['Combination'] == '5_AllFeatures+Engineering')
]['CV_RMSE'].values[0]

print(f"\n📊 Comparison:")
print(f"   Baseline Ridge (alpha=1.0): {baseline_ridge:.4f}")
print(f"   Tuned Ridge (alpha=7.67): 0.1111")
print(f"   Tuned ElasticNet: {elasticnet_rmse:.4f}")
print(f"   Best Ensemble: {best_model['CV_RMSE']:.4f}")

improvement_from_baseline = (baseline_ridge - best_model['CV_RMSE']) / baseline_ridge * 100
improvement_from_ridge = (0.1111 - best_model['CV_RMSE']) / 0.1111 * 100

print(f"\n📈 Improvements:")
print(f"   vs Baseline Ridge: {improvement_from_baseline:.2f}%")
print(f"   vs Tuned Ridge: {improvement_from_ridge:.2f}%")

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
colors = ['gold' if i == 0 else 'green' if i == 1 else 'steelblue' if i < 3 else 'lightblue' 
          for i in range(len(results_df_final))]
plt.barh(range(len(results_df_final)), results_df_final['CV_RMSE'], color=colors)
plt.yticks(range(len(results_df_final)), results_df_final['Model/Ensemble'])
plt.xlabel('CV RMSE (log scale)')
plt.title('Model & Ensemble Performance Comparison (Best Linear Model)', fontweight='bold')
plt.axvline(x=baseline_ridge, color='red', linestyle='--', linewidth=2, label='Baseline Ridge (0.1132)')
plt.axvline(x=0.1111, color='orange', linestyle='--', linewidth=2, label='Tuned Ridge (0.1111)')
plt.legend()
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

# Add values on bars
for i, (idx, row) in enumerate(results_df_final.iterrows()):
    plt.text(row['CV_RMSE'] + 0.001, i, f"{row['CV_RMSE']:.4f}", 
             va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Training and ensembling complete!")
print(f"\n🎉 ElasticNet improved over Ridge by {improvement_from_ridge:.2f}%!")

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

# 9. Submit your predictions to the competition

In [ ]:
print("Creating Submission File")
print("="*70)

# Get best model from results
best_model_name = results_df_final.iloc[0]['Model/Ensemble']
best_cv_rmse = results_df_final.iloc[0]['CV_RMSE']

print(f"\nBest Model: {best_model_name}")
print(f"CV RMSE: {best_cv_rmse:.4f}")

# Get test data
X_test_final = feature_combinations['5_AllFeatures+Engineering']['X_test'].fillna(0)
print(f"\nTest data shape: {X_test_final.shape}")

# Train on FULL training data (no CV)
print("\nTraining on full training set...")
elasticnet_optimized.fit(X_train_final, y_train_log)
dt_optimized.fit(X_train_final, y_train_log)

# Predict on test set
elasticnet_test_pred = elasticnet_optimized.predict(X_test_final)
dt_test_pred = dt_optimized.predict(X_test_final)

# Determine ensemble weights from best model
if 'ElasticNet (optimized)' in best_model_name:
    ensemble_pred_log = elasticnet_test_pred
    weight_desc = "ElasticNet only"
elif '95/5' in best_model_name:
    ensemble_pred_log = 0.95 * elasticnet_test_pred + 0.05 * dt_test_pred
    weight_desc = "95% ElasticNet + 5% DT"
elif '90/10' in best_model_name:
    ensemble_pred_log = 0.9 * elasticnet_test_pred + 0.1 * dt_test_pred
    weight_desc = "90% ElasticNet + 10% DT"
elif '80/20' in best_model_name:
    ensemble_pred_log = 0.8 * elasticnet_test_pred + 0.2 * dt_test_pred
    weight_desc = "80% ElasticNet + 20% DT"
elif '70/30' in best_model_name:
    ensemble_pred_log = 0.7 * elasticnet_test_pred + 0.3 * dt_test_pred
    weight_desc = "70% ElasticNet + 30% DT"
elif '60/40' in best_model_name:
    ensemble_pred_log = 0.6 * elasticnet_test_pred + 0.4 * dt_test_pred
    weight_desc = "60% ElasticNet + 40% DT"
elif 'Simple Average' in best_model_name:
    ensemble_pred_log = 0.5 * elasticnet_test_pred + 0.5 * dt_test_pred
    weight_desc = "50% ElasticNet + 50% DT"
else:
    ensemble_pred_log = dt_test_pred
    weight_desc = "Decision Tree only"

print(f"Using: {weight_desc}")

# Convert back to original scale
ensemble_pred_dollars = np.expm1(ensemble_pred_log)

print(f"✓ Predictions generated for {len(ensemble_pred_dollars)} houses")

# Create submission dataframe
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': ensemble_pred_dollars
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

print(f"\n✓ Submission file created: submission_best.csv")
print(f"\nFirst 5 predictions:")
print(submission.head())

print(f"\nPrediction Statistics:")
print(f"  Mean: ${ensemble_pred_dollars.mean():,.0f}")
print(f"  Median: ${np.median(ensemble_pred_dollars):,.0f}")
print(f"  Min: ${ensemble_pred_dollars.min():,.0f}")
print(f"  Max: ${ensemble_pred_dollars.max():,.0f}")

print("\n" + "="*70)
print("🎉 Ready to submit to Kaggle!")
print(f"File: submission_best.csv")
print(f"Model: {best_model_name}")
print(f"Expected RMSE: ~{best_cv_rmse:.4f}")

In [ ]:
%%bash

cat submission.csv | head -n 5